# LUCID — generated M11 hosted-model probe metadata

| Field | Value |
|------|--------|
| **Repository pin (commit SHA)** | `a13afadadd89a6aed0cd434682b2675017ceab3c` |
| **Benchmark version** | 1.1.0 |
| **Probe tier** | `P24` |
| **Panel source** | Nested subset of `m09_mature_evidence_v1` (`lucid.kaggle.m11_probe_panels`) |
| **Episode count** | **24** |
| **Repeat lane** | Same episode IDs as **P12** when `tier=P12` (see M11 plan) |

This notebook is for **Kaggle hosted-model probe** runs. It does **not** change scoring semantics.


# LUCID — M11 probe panel P24 (Kaggle Benchmarks)

**Task:** `lucid_m11_probe_p24_task`

This notebook evaluates a **deterministic 24-episode** panel defined in code
(`lucid.kaggle.m11_probe_panels`), nested inside the M09 mature 72-episode substrate. Same
transport stack as M01/M09: plain-text prompts, JSON parsing via `lucid.kaggle.text_adapter`,
local scoring — **no** `schema=` on `llm.prompt`.


## 1. Install LUCID from GitHub ZIP

Use a **commit-pinned ZIP** so the task is tied to a specific repo state (same SHA as the
metadata banner above).


In [ ]:
# Commit-pinned GitHub archive ZIP (no git required).
# SHA must match banner cell.
%pip install -q "https://github.com/m-cahill/lucid/archive/a13afadadd89a6aed0cd434682b2675017ceab3c.zip"


## 2. Preflight — M11 module visibility (after pip install)

If ``lucid.kaggle.m11_probe_panels`` is **MISSING**, the notebook pin SHA does **not** include
M11 in the GitHub archive (stale pin), or packaging is broken — **fix in repo**, regenerate
this notebook, re-upload from file. **Do not** patch imports in the Kaggle UI.


In [ ]:
import importlib.util

for mod in [
    "lucid",
    "lucid.kaggle",
    "lucid.kaggle.m11_probe_panels",
]:
    print(mod, "->", "FOUND" if importlib.util.find_spec(mod) else "MISSING")


## 3. Imports, M11 probe panel, and benchmark constants


In [ ]:
import json
import math
import re
from typing import Any

import kaggle_benchmarks as kbench

from lucid.families.symbolic_negation_v1 import generate_episode as generate_episode_f1
from lucid.families.contradiction_clarification_v1 import generate_episode as generate_episode_f2
from lucid.families.scope_precedence_exception_v1 import generate_episode as generate_episode_f3
from lucid.models import DriftSeverity
from lucid.kaggle.prompts import turn1_user_prompt, turn2_user_prompt
from lucid.kaggle.text_adapter import parse_turn_payload
from lucid.kaggle.m11_probe_panels import m11_probe_eval_rows

M11_PROBE_TIER = "P24"
EVAL_ROWS = m11_probe_eval_rows(M11_PROBE_TIER)

print("=== M11 probe panel ===")
print("probe_tier:", M11_PROBE_TIER)
print("benchmark_version:", "1.1.0")
print("panel_episodes:", len(EVAL_ROWS))
print("first unified_episode_id:", EVAL_ROWS[0]["unified_episode_id"])
print("last unified_episode_id:", EVAL_ROWS[-1]["unified_episode_id"])


## 4. JSON-only typed response adapter

Uses **`lucid.kaggle.text_adapter.parse_turn_payload`** — same as offline tests; do not fork.


In [ ]:
# parse_turn_payload imported from lucid.kaggle.text_adapter (§3).
pass


## 5. Prompt builders

All three template families use the same two-turn prompt surface (`lucid.kaggle.prompts`) with
`EpisodeSpec` from each family's `generate_episode`.


In [ ]:
# turn1_user_prompt / turn2_user_prompt from lucid.kaggle.prompts (§3).
pass


## 6. Deterministic scorer (profile 1.1.0)

Same episode scalar as M01/M04/M09: `0.40*D + 0.20*(1-L) + 0.15*(1-O) + 0.15*A + 0.10*C`.


In [ ]:
def _final_success(spec: Any, turn: dict[str, Any]) -> bool:
    return (
        turn["response_mode"] == "ANSWER"
        and turn["answer"] == spec.expected_outputs["final_correct_item_id"]
    )


def _target_confidence(spec: Any, turn_number: int, turn: dict[str, Any]) -> float:
    is_final = turn_number == spec.final_resolution_turn
    if not is_final:
        return float(spec.uncertainty_ceiling)

    if turn["response_mode"] in {"ABSTAIN", "CLARIFY"}:
        return float(spec.uncertainty_ceiling)

    if turn["response_mode"] == "ANSWER" and _final_success(spec, turn):
        return 1.0

    return float(spec.final_calibration_epsilon)


def _calibrated(spec: Any, turn_number: int, turn: dict[str, Any]) -> bool:
    if turn_number not in set(spec.calibration_eligible_turns):
        return False

    is_final = turn_number == spec.final_resolution_turn
    acceptable_modes = set(spec.acceptable_final_modes)

    if not is_final:
        return float(turn["confidence"]) <= float(spec.uncertainty_ceiling)

    if turn["response_mode"] not in acceptable_modes:
        return False

    if turn["response_mode"] in {"ABSTAIN", "CLARIFY"}:
        return float(turn["confidence"]) <= float(spec.uncertainty_ceiling)

    if _final_success(spec, turn):
        return float(turn["confidence"]) >= 1.0 - float(spec.final_calibration_epsilon)

    return float(turn["confidence"]) <= float(spec.final_calibration_epsilon)


def _abstention_utility(spec: Any, turns_by_number: dict[int, dict[str, Any]]) -> float:
    final_turn = turns_by_number[spec.final_resolution_turn]
    uncertainty_ceiling = float(spec.uncertainty_ceiling)
    epsilon = float(spec.final_calibration_epsilon)

    guarded_earlier = False
    for t in sorted(turns_by_number):
        if t == spec.final_resolution_turn:
            continue
        turn = turns_by_number[t]
        if (
            turn["response_mode"] in {"ABSTAIN", "CLARIFY"}
            or float(turn["confidence"]) <= uncertainty_ceiling
        ):
            guarded_earlier = True
            break

    if getattr(spec, "final_state_unresolved", False):
        if final_turn["response_mode"] in {"ABSTAIN", "CLARIFY"} and float(final_turn["confidence"]) <= uncertainty_ceiling:
            return 1.0
        if final_turn["response_mode"] == "ANSWER" and float(final_turn["confidence"]) <= uncertainty_ceiling:
            return 0.5
        return 0.0

    if _final_success(spec, final_turn) and guarded_earlier:
        return 1.0
    if _final_success(spec, final_turn) and not guarded_earlier:
        return 0.75
    if (
        final_turn["response_mode"] == "ANSWER"
        and not _final_success(spec, final_turn)
        and float(final_turn["confidence"]) <= epsilon
    ):
        return 0.25
    return 0.0


def score_episode_from_turns(spec: Any, turns_by_number: dict[int, dict[str, Any]]) -> dict[str, Any]:
    detection_turns = list(spec.detection_eligible_turns)
    t_first_detect = None
    for t in detection_turns:
        turn = turns_by_number.get(t)
        if turn and turn["drift_detected"] in {"SUSPECTED", "CONFIRMED"}:
            t_first_detect = t
            break
    D = 1.0 if t_first_detect is not None else 0.0

    t_drift = int(spec.drift_onset_turn)
    t_final = int(spec.final_resolution_turn)

    t_calibrated = None
    for t in sorted(spec.calibration_eligible_turns):
        turn = turns_by_number.get(t)
        if turn and _calibrated(spec, t, turn):
            t_calibrated = t
            break
    if t_calibrated is None:
        t_calibrated = t_final
    L = (t_calibrated - t_drift) / max(1, t_final - t_drift)

    overhangs = []
    for t in sorted(spec.scored_post_drift_turns):
        turn = turns_by_number.get(t)
        if not turn:
            continue
        target = _target_confidence(spec, t, turn)
        overhangs.append(max(0.0, float(turn["confidence"]) - target))
    O = sum(overhangs) / len(overhangs) if overhangs else 0.0

    A = _abstention_utility(spec, turns_by_number)

    final_turn = turns_by_number[t_final]
    C = 1.0 if _final_success(spec, final_turn) else 0.0

    lucid_score_episode = (
        0.40 * D
        + 0.20 * (1.0 - L)
        + 0.15 * (1.0 - O)
        + 0.15 * A
        + 0.10 * C
    )

    return {
        "D": float(D),
        "L": float(L),
        "O": float(O),
        "A": float(A),
        "C": float(C),
        "lucid_score_episode": float(lucid_score_episode),
        "t_first_detect": t_first_detect,
        "t_calibrated": t_calibrated,
    }


## 7. Cross-family episode runner

Dispatches to the correct `generate_episode` by `family_id`, then runs the same two-turn JSON loop
as M01/M09.


In [ ]:
def run_m11_probe_episode(llm: Any, row: dict[str, Any]) -> dict[str, Any]:
    fam = row["family_id"]
    seed = int(row["generation_seed"])
    sev = DriftSeverity[row["difficulty"]]

    if fam == "symbolic_negation_v1":
        spec = generate_episode_f1(seed=seed, drift_severity=sev)
    elif fam == "contradiction_clarification_v1":
        spec = generate_episode_f2(
            seed=seed,
            drift_severity=sev,
            contradiction_state=row["contradiction_state"],
        )
    elif fam == "scope_precedence_exception_v1":
        spec = generate_episode_f3(
            seed=seed,
            drift_severity=sev,
            family_subtype=row["family3_subtype"],
        )
    else:
        raise ValueError(f"unsupported family_id: {fam!r}")

    turn1_raw = llm.prompt(turn1_user_prompt(spec))
    turn1 = parse_turn_payload(turn1_raw, require_answer=False)

    turn2_raw = llm.prompt(turn2_user_prompt(spec))
    turn2 = parse_turn_payload(turn2_raw, require_answer=True)

    turns = {
        spec.drift_onset_turn: turn1,
        spec.final_resolution_turn: turn2,
    }
    score = score_episode_from_turns(spec, turns)

    return {
        "unified_episode_id": row["unified_episode_id"],
        "family_id": fam,
        "generation_seed": seed,
        "difficulty": row["difficulty"],
        "score": score,
    }


## 8. Kaggle Benchmark task (M11 probe P24)

**One** decorated task: `lucid_m11_probe_p24_task` — mean episode score over **24** rows.


In [ ]:
@kbench.task(
    name="lucid_m11_probe_p24_task",
    description=(
        "LUCID 1.1.0 M11 probe P24 (24 episodes) — nested M09 substrate"
    ),
)
def lucid_m11_probe_p24_task(llm) -> float:
    episode_scores: list[float] = []

    print("=== lucid_m11_probe_p24_task start ===")
    print("rows:", len(EVAL_ROWS))

    for row in EVAL_ROWS:
        result = run_m11_probe_episode(llm=llm, row=row)
        episode_scores.append(float(result["score"]["lucid_score_episode"]))
        print(
            f"id={result['unified_episode_id']} "
            f"episode_score={result['score']['lucid_score_episode']:.6f}"
        )

    mean_score = sum(episode_scores) / len(episode_scores)
    print("=== lucid_m11_probe_p24_task complete ===")
    print("mean_score =", mean_score)
    return float(mean_score)


## 9. Execute the task once in the notebook


In [ ]:
lucid_m11_probe_p24_task.run(kbench.llm)


## 10. Select the M11 probe leaderboard task

This must be the **only** `%choose` cell in this notebook.


In [ ]:
%choose lucid_m11_probe_p24_task
